# 🧪 LAB: Machine Learning to Predict Concrete Compressive Strength

In this lab, you will apply `PyTorch` to a real-world data problem. You will implement a Neural Network in `PyTorch` using the new functionalities introduced this week and compare its performance with a Support Vector Machine and Linear Regression.

Your goal is to predict the **compressive strength of concrete** (the outcome variable, *Y*) based on a set of input features (*X*), including:
**Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate, and Age.**

This dataset and problem are based on the following paper:

- I-Cheng Yeh, "Modeling of strength of high performance concrete using artificial neural networks," Cement and Concrete Research, Vol. 28, No. 12, pp. 1797-1808 (1998). [Link to the paper](yeh1998.pdf).


---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

NAME(s) = "Ryan, Adalia, Josh"

## 0. Overall Instructions 

In this lab, you will work with three models: a **Neural Network**, a **Support Vector Machine (SVM)**, and some form of **Linear Regression**. Each should meet the following requirements:

- **Do not tune the models**. For example, for the Neural Network, choose any number of layers and hidden units you prefer; for the SVM, select a kernel of your choice (e.g., linear, RBF, polynomial), etc. For example, when using GridSearchCV to find the best value for a hyperparameter out of 5 values, that would be considered tuning the hyperparameters which is what we want to avoid here.

- **Evaluate model performance using k-fold cross-validation**. That is, run each model using a chosen number of folds (*k*), and report the average performance across all folds. You may select the value of *k* that you want.

- **Be careful with data preprocessing**. Apply all preprocessing steps properly to avoid data leakage (e.g. standardizing features before data splitting)

## 1. Pre-implementation Group Discussion (15 points)

In your group, discuss, agree on, and elaborate the following points:

- **Descriptive analysis**. Identify what exploratory or descriptive analyses you can perform to better understand the relationship between each input variable and the outcome. Consider both visual (e.g., scatter plots, histograms) and statistical summaries.

- **Data preprocessing**. Decide what preprocessing steps are necessary given that you will be using neural networks. 

- **Model configuration**. Specify which cost function and output layer activation are most appropriate for your problem type (e.g., regression vs. classification). Explain your reasoning based on the nature of the outcome variable.

- **Performance evaluation**. Describe how you will perform k-fold cross-validation to assess your model’s performance. Include how you will divide the data, compute metrics across folds, and report the final averaged results.

1. Descriptive Analysis

We will create scatter plots of each feature to visualize the distribution of the data and whether any corrective actions like log transformations are necessary. In addition, we will find the mean, median, maximum, and minimum of each feature, which will help spot outliers or skewness. 
   
2. Data Preprocessing

The main preprocessing steps we will do are doing a k-fold split for cross-validation where each split can serve as the test set once, standardizing the data, and possibly one hot encoding or ordinal encoding if there are any categorical features or if the target has multiple classes.

3. Model Configuration

We will use a Mean Squared Error cost function and no output layer activation because we have a regression problem type. Mean Squared Error is a good cost function for regression models because the outcome variable will be continuous and the target feature is continuous, so this is one of the best ways to measure performance by finding the difference between the predicted target and the true target and squaring it.

4. Performance Evaluation

We will perform k-fold cross-validation with k=5 so that the data is split into 5 components, where each set will behave as the test set once while the other 4 act as the training set. We will use cross_val_score to compute and return MSE for each k and k-fold cross validation will automatically return the final averaged MSE from all splits.

## 2. Descriptive analysis (10 points)

Apply the descriptive analyses that your group agreed upon in the previous section.

**Load the dataset before beginning the descriptive analysis; you will also use it in the following sections.**

In [1]:
# import packages
import numpy as np
import pandas as pd

In [7]:
# Used this pip install to use .read_excel()
# !pip install xlrd

In [5]:
df = pd.read_excel("Concrete_Data.xls")
df.head()

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


## 3. Neural Network (40 points)

Implement a neural network as you did in the previous lab, but this time leverage the new functionalities introduced this week. Specifically:

- Your class implementing the neural network should inherit from nn.Module.

- Use `nn.Linear` for the linear transformations between layers, `nn.ReLU` for the hidden-layer activations, and the appropriate activation function for the output layer (e.g., `nn.Sigmoid` for binary classification).

- Use the appropriate cost function from `torch.nn` for your task (e.g., `nn.MSELoss`, `nn.BCELoss`).

- Use Adam from `torch.optim` as the optimizer.

- Use `DataLoader` to efficiently do batch processing. You may choose the batch size freely.

Once you have implemented this class, create an instance of it, then train and evaluate your model on the lab dataset.

**Remember to follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.**

Notes
- hidden_dims will be a list like [6] or [12,12] where one value is for one layer and 2 values is for 2 layers

In [19]:
# importing packages
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [29]:
class NeuralNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[10], learning_rate=0.01):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.learning_rate = learning_rate
        self.module_list = nn.ModuleList(layers)
        self.mse = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def forward(self, X):
        for f in self.module_list:
            X = f(X)
        return X

    def cost(self, y_pred, y_true):
        return self.mse(y_pred, y_true.view(-1,1))

    def fit(self, data_loader, n_epochs=100):
        self.train()
        for epoch in range(n_epochs):
            for x_batch, y_batch in data_loader:
                self.optimizer.zero_grad()
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()
                self.optimizer.step()

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            return self.forward(X)

In [30]:
X = np.array(df.iloc[:,0:8])
y = np.array(df.iloc[:,-1])

In [31]:
# Define batch size
batch_size = 16

In [33]:
# Implement a for loop with scikit-learn k-fold where it trains and predicts for each loop
# Calls the class inside the loop
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(kf.split(X_train)):
    # spltting data based on folds so each of the 5 folds is the test fold once
    X_train_batch, y_train_batch = X_train[train_idx], y_train[train_idx]
    X_test_batch, y_test_batch = X_train[test_idx], y_train[test_idx]
    # create a fold dataloader
    fold_dataset = TensorDataset(X_train_batch, y_train_batch)
    fold_loader = DataLoader(fold_dataset, batch_size=batch_size, shuffle=True)
    
    torch.manual_seed(42)
    model = NeuralNet(input_dim=X_train.shape[1])
    model.fit(fold_loader, n_epochs=100)
    y_preds = model.predict(X_test_batch)
    fold_mse = model.cost(y_preds, y_test_batch)
    print(f'Fold: {fold+1}, MSE: {fold_mse:.5f}')

Fold: 1, MSE: 104.04456
Fold: 2, MSE: 115.53707
Fold: 3, MSE: 88.52012
Fold: 4, MSE: 103.31017
Fold: 5, MSE: 97.16010


## 4. Replication (10 points)

Replicate your neural network experiment using the corresponding model class from `scikit-learn`: https://scikit-learn.org/stable/api/sklearn.neural_network.html. Be sure to apply the same neural network architecture as well as the same procedures for model evaluation and preprocessing to ensure a fair comparison.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

In [5]:
# USE AS MANY CELLS AS NEEDED

## 5. Comparison (20 points)

Apply a Support Vector Machine (SVM) and Linear Regression model to the dataset, and compare their performance with that of your Neural Network.

Discuss any similarities or differences you observe in their results and provide possible explanations for these patterns.

Be sure to apply the k-fold cross-validation procedure to ensure a fair comparison across all models.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

In [6]:
# USE AS MANY CELLS AS NEEDED

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE